In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

In [17]:
indo_stock = pd.read_csv('price_last_capchg.csv')
indo_stock = indo_stock.set_index('date')
indo_stock_returns = indo_stock.pct_change()
indo_stock_returns.index = pd.to_datetime(indo_stock_returns.index)

In [129]:
indo_stock_returns

,AADI,AALI,ABBA,ABDA,ABMM,ACAP,ACES,ACRO,ACST,ADCP,...,YELO,YOII,YPAS,YULE,YUPI,ZATA,ZBRA,ZINC,ZONE,ZYRX
date,,,,,,,,,,,,,,,,,,,,,
2002-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2002-01-03,NaN,0.027778,NaN,0.000000,NaN,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2002-01-04,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2002-01-07,NaN,0.054054,NaN,0.000000,NaN,-0.080000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2002-01-08,NaN,-0.051282,NaN,0.000000,NaN,-0.057971,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-30,-0.022013,-0.004184,0.0,0.000000,-0.008621,NaN,-0.018072,-0.032258,0.000000,0.0,...,-0.059701,-0.033708,-0.041322,-0.014620,-0.050584,0.0,0.0,0.0,-0.110236,-0.007692
2026-07-01,0.012862,0.004202,0.0,0.000000,-0.021739,NaN,-0.006135,-0.033333,0.000000,0.0,...,0.111111,0.000000,-0.008621,-0.002967,-0.004098,0.0,0.0,0.0,0.017699,-0.007752
2026-07-02,0.003175,0.000000,0.0,0.000000,0.004444,NaN,0.012346,0.000000,0.000000,0.0,...,-0.028571,-0.011628,0.034783,-0.041667,-0.004115,0.0,0.0,0.0,0.017391,0.000000


In [116]:
invuni = pd.read_parquet('invuni_v2.parquet')

### Functions

#### Filter by Universe

In [6]:
def filter_by_universe(
    sentiment_df: pd.DataFrame,
    universe_df: pd.DataFrame,
    cutoff_date: str = "2026-06-15",
) -> pd.DataFrame:

    sentiment = sentiment_df.copy()
    universe = universe_df.copy()

    if "date" not in universe.columns:
        universe = universe.reset_index()

    universe = universe.rename(
        columns={"stock": "ticker"}
    )

    sentiment["ticker"] = (
        sentiment["ticker"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    universe["ticker"] = (
        universe["ticker"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    sentiment["date"] = pd.to_datetime(
        sentiment["date"],
        errors="coerce",
    ).dt.normalize()

    universe["date"] = pd.to_datetime(
        universe["date"],
        errors="coerce",
    ).dt.normalize()

    cutoff = pd.Timestamp(cutoff_date)

    eligible_universe = (
        universe.loc[
            universe["uni"].eq(1),
            ["date", "ticker"],
        ]
        .dropna()
        .drop_duplicates()
    )

    eligible_pairs = pd.MultiIndex.from_frame(
        eligible_universe[["date", "ticker"]]
    )

    sentiment_pairs = pd.MultiIndex.from_frame(
        sentiment[["date", "ticker"]]
    )

    keep_mask = (
        sentiment["date"].gt(cutoff)
        | sentiment_pairs.isin(eligible_pairs)
    )

    filtered = sentiment.loc[keep_mask].copy()

    return filtered

#### JSON to CSV

In [49]:
def json_to_csv(
    json_file: str,
    csv_file: str | None = None,
) -> pd.DataFrame:
    
    ''' 
    Converts JSON to CSV and add one column called the weighted sentiment by multiplying the 3 factors that exists
    '''
    
    articles = pd.read_json(json_file)

    rows = [
        stock
        for article_stocks in articles["stocks"]
        for stock in article_stocks
    ]

    sentiment_df = pd.DataFrame(rows)

    if not sentiment_df.empty and "date" in sentiment_df.columns:
        sentiment_df["date"] = pd.to_datetime(
            sentiment_df["date"]
        )

    if csv_file is not None:
        sentiment_df.to_csv(
            csv_file,
            index=False,
            encoding="utf-8-sig",
        )

    sentiment_df['weighted_sentiment'] = sentiment_df['sentiment_score'] * sentiment_df['relevance'] * sentiment_df['confidence']

    return sentiment_df

#### T-test summary

In [51]:
def stat_summary(table_df: pd.DataFrame) -> pd.DataFrame:

    table_describe = table_df.describe(include='all')

    count = pd.to_numeric(table_describe.loc['count'], errors='coerce')
    mean  = pd.to_numeric(table_describe.loc['mean'],  errors='coerce')
    std   = pd.to_numeric(table_describe.loc['std'],   errors='coerce')

    t_stat = mean / (std / np.sqrt(count))

    table_describe.loc['t-stat'] = t_stat

    return table_describe

#### Simulation Strategy #1

In [134]:
'''
Here, we take the top 10th/bottom 10th percentile and long the top 10, whilst shorting the bottom 10. To add on, we use
percentile_no to calculate for a given top/bottom percentile.
'''
def strat_long_short(data: pd.DataFrame, stock: pd.DataFrame, filter: str='avg_weighted_sentiment', percentile_no: float=0.1) -> pd.DataFrame:
    
    data = data.copy()
    if 'date' in data.columns:
        data = data.set_index('date')
        
    data.index = pd.to_datetime(data.index)
    stock.index = pd.to_datetime(stock.index)

    return_dataframe = []
    
    for date_idx, daily_data in data.groupby(data.index):
        
        daily_data = daily_data.sort_values(by=filter)
        
        cutoff = int(len(daily_data) * percentile_no)
        
    
        if cutoff == 0:
            continue 
            
        bottom_10 = daily_data.head(cutoff) 
        top_10 = daily_data.tail(cutoff) 

        for _, row in top_10.iterrows():
            ticker = row.ticker
            
            if ticker in stock.columns:
                mini_df = stock[ticker].loc[date_idx:].iloc[:3]
                
                if len(mini_df) >= 3:
                    return_dataframe.append({
                        "ticker": ticker,
                        "start_date": mini_df.index[0], 
                        "date_after": mini_df.index[1],
                        "return": mini_df.iloc[2],
                        "long/short": 1,
                    })
        
        for _, row in bottom_10.iterrows():
            ticker = row.ticker
            
            if ticker in stock.columns:
                mini_df = stock[ticker].loc[date_idx:].iloc[:3]
                
                if len(mini_df) >= 3:
                    return_dataframe.append({
                        "ticker": ticker,
                        "start_date": mini_df.index[0], 
                        "date_after": mini_df.index[1],
                        "return": mini_df.iloc[2],
                        "long/short": -1,
                    })

    return pd.DataFrame(return_dataframe)

In [ ]:
'''
Here, we take the top 10th/bottom 10th percentile and long the top 10, whilst shorting the bottom 10. To add on, we use
percentile_no to calculate for a given top/bottom percentile. 
Same thing, but we short the positive sentiment and long the negative sentiment
'''
def reverse_strat_long_short(data: pd.DataFrame, stock: pd.DataFrame, filter: str='avg_weighted_sentiment', percentile_no: float=0.1) -> pd.DataFrame:
    
    data = data.copy()
    if 'date' in data.columns:
        data = data.set_index('date')
        
    data.index = pd.to_datetime(data.index)
    stock.index = pd.to_datetime(stock.index)

    return_dataframe = []
    
    for date_idx, daily_data in data.groupby(data.index):
        
        daily_data = daily_data.sort_values(by=filter)
        
        cutoff = int(len(daily_data) * percentile_no)
        
    
        if cutoff == 0:
            continue 
            
        bottom_10 = daily_data.head(cutoff) 
        top_10 = daily_data.tail(cutoff) 

        for _, row in top_10.iterrows():
            ticker = row.ticker
            
            if ticker in stock.columns:
                mini_df = stock[ticker].loc[date_idx:].iloc[:3]
                
                if len(mini_df) >= 3:
                    return_dataframe.append({
                        "ticker": ticker,
                        "start_date": mini_df.index[0], 
                        "date_after": mini_df.index[1],
                        "return": mini_df.iloc[2],
                        "long/short": -1,
                    })
        
        for _, row in bottom_10.iterrows():
            ticker = row.ticker
            
            if ticker in stock.columns:
                mini_df = stock[ticker].loc[date_idx:].iloc[:3]
                
                if len(mini_df) >= 3:
                    return_dataframe.append({
                        "ticker": ticker,
                        "start_date": mini_df.index[0], 
                        "date_after": mini_df.index[1],
                        "return": mini_df.iloc[2],
                        "long/short": 1,
                    })

    return pd.DataFrame(return_dataframe)

#### Simulation Strategy #2

In [170]:
''' 
Similar as to Strategy #1, but here we use the EWMA
'''

def make_ewma(data: pd.DataFrame, filter: str = 'avg_weighted_sentiment') -> pd.DataFrame:

    data = data.sort_values(
        ["ticker", "date"]
    ).copy()       

    data["ewma_sentiment"] = (
    data
    .groupby("ticker")[filter]
    .transform(
        lambda x: x.ewm(
            halflife=5,
            adjust=False,
            min_periods=1,
        ).mean()
        )
    ) 

    return data


### Data Analysis

In [9]:
data = json_to_csv('all_articles.json')[['ticker','relevance','sentiment_score','confidence','date','weighted_sentiment']]
data = data.drop(data[data['sentiment_score']==0].index)

In [10]:
sentiment_df = data.groupby(['ticker', 'date']).agg(
        mention_count=('ticker', 'count'),
        avg_sentiment=('sentiment_score', 'mean'),
        avg_weighted_sentiment=('weighted_sentiment', 'mean')
    ).reset_index()

In [11]:
sentiment_df = filter_by_universe(sentiment_df, invuni)

In [12]:
# Main overview of all the data that we are dealing with
sentiment_df['total_sentiment'] = sentiment_df['avg_sentiment']*sentiment_df['mention_count']

In [13]:
percentiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9,0.95]
sentiment_df['mention_count'].quantile(percentiles)

0.10     1.0
0.20     1.0
0.30     1.0
0.40     1.0
0.50     2.0
0.60     2.0
0.70     3.0
0.80     4.0
0.90     7.0
0.95    12.0
Name: mention_count, dtype: float64

In [14]:
# Take bottom 10 percentile and top 10 percentile from here
sentiment_df

,ticker,date,mention_count,avg_sentiment,avg_weighted_sentiment,total_sentiment
0,EMTK,2025-12-04,1,0.500000,0.281250,0.50
1,MIKA,2026-06-19,1,-0.650000,-0.487500,-0.65
2,AADI,2025-07-14,1,0.500000,0.375000,0.50
3,AADI,2025-07-17,1,0.500000,0.375000,0.50
4,AADI,2025-07-23,3,0.433333,0.305000,1.30
...,...,...,...,...,...,...
27779,ZONE,2026-06-18,3,0.616667,0.384375,1.85
27780,ZONE,2026-06-19,2,0.750000,0.556875,1.50
27781,ZONE,2026-06-22,3,0.800000,0.500000,2.40
27782,ZONE,2026-06-23,3,-0.616667,-0.415625,-1.85


### Backtest

In [135]:
sentiment_df = sentiment_df[sentiment_df['date']<'2026-06-16']

#### Strat Testing #1

Take top/bottom 10 percentile, long/short them. Percentile is based off three factors in which all of them is tested, namely: *avg_weighted_sentiment*, *total_sentiment*, *avg_sentiment*.

In [136]:
strat_aws= strat_long_short(sentiment_df,indo_stock_returns)
strat_aws['profits'] = strat_aws['long/short']*strat_aws['return']
stat_summary(strat_aws)

,ticker,start_date,date_after,return,long/short,profits
count,3504,3504,3504,3504.000000,3504.000000,3504.000000
unique,299,NaN,NaN,NaN,NaN,NaN
top,DEWA,NaN,NaN,NaN,NaN,NaN
freq,52,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 18:47:40.273972,2025-12-24 07:01:38.630137,0.000388,0.000000,0.000856
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.150000,-1.000000,-0.349315
25%,NaN,2025-10-02 18:00:00,2025-10-05 06:00:00,-0.020313,-1.000000,-0.017544
50%,NaN,2025-12-15 12:00:00,2025-12-16 12:00:00,0.000000,0.000000,0.000000
75%,NaN,2026-03-11 00:00:00,2026-03-12 00:00:00,0.015385,1.000000,0.018382
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.349315,1.000000,0.343511


In [137]:
strat_ts= strat_long_short(sentiment_df,indo_stock_returns,'total_sentiment')
strat_ts['profits'] = strat_ts['long/short']*strat_ts['return']
stat_summary(strat_ts)

,ticker,start_date,date_after,return,long/short,profits
count,3504,3504,3504,3504.000000,3504.000000,3504.000000
unique,235,NaN,NaN,NaN,NaN,NaN
top,BBRI,NaN,NaN,NaN,NaN,NaN
freq,168,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 18:47:40.273972,2025-12-24 07:01:38.630137,-0.000998,0.000000,-0.000315
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.150000,-1.000000,-0.349315
25%,NaN,2025-10-02 18:00:00,2025-10-05 06:00:00,-0.018349,-1.000000,-0.016678
50%,NaN,2025-12-15 12:00:00,2025-12-16 12:00:00,-0.002558,0.000000,0.000000
75%,NaN,2026-03-11 00:00:00,2026-03-12 00:00:00,0.013333,1.000000,0.016227
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.349315,1.000000,0.248731


In [139]:
strat_as= strat_long_short(sentiment_df,indo_stock_returns,'avg_sentiment')
strat_as['profits'] = strat_as['long/short']*strat_as['return']
stat_summary(strat_as)

,ticker,start_date,date_after,return,long/short,profits
count,3504,3504,3504,3504.000000,3504.000000,3504.000000
unique,298,NaN,NaN,NaN,NaN,NaN
top,DSSA,NaN,NaN,NaN,NaN,NaN
freq,51,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 18:47:40.273972,2025-12-24 07:01:38.630137,0.000972,0.000000,0.001023
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.150000,-1.000000,-0.250000
25%,NaN,2025-10-02 18:00:00,2025-10-05 06:00:00,-0.020000,-1.000000,-0.018182
50%,NaN,2025-12-15 12:00:00,2025-12-16 12:00:00,0.000000,0.000000,0.000000
75%,NaN,2026-03-11 00:00:00,2026-03-12 00:00:00,0.016353,1.000000,0.018388
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.343511,1.000000,0.343511


In [140]:
# Long only strategy as we 
stat_summary(strat_as[strat_as['long/short']==1])

,ticker,start_date,date_after,return,long/short,profits
count,1752,1752,1752,1752.000000,1752.0,1752.000000
unique,264,NaN,NaN,NaN,NaN,NaN
top,PGEO,NaN,NaN,NaN,NaN,NaN
freq,24,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 18:47:40.273972,2025-12-24 07:01:38.630137,0.001995,1.0,0.001995
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.149718,1.0,-0.149718
25%,NaN,2025-10-02 18:00:00,2025-10-05 06:00:00,-0.019608,1.0,-0.019608
50%,NaN,2025-12-15 12:00:00,2025-12-16 12:00:00,0.000000,1.0,0.000000
75%,NaN,2026-03-11 00:00:00,2026-03-12 00:00:00,0.016814,1.0,0.016814
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.343511,1.0,0.343511


In [141]:
stat_summary(strat_as[strat_as['long/short']==-1])

,ticker,start_date,date_after,return,long/short,profits
count,1752,1752,1752,1752.000000,1752.0,1752.000000
unique,248,NaN,NaN,NaN,NaN,NaN
top,CUAN,NaN,NaN,NaN,NaN,NaN
freq,37,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 18:47:40.273972,2025-12-24 07:01:38.630137,-0.000051,-1.0,0.000051
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.150000,-1.0,-0.250000
25%,NaN,2025-10-02 18:00:00,2025-10-05 06:00:00,-0.020386,-1.0,-0.015687
50%,NaN,2025-12-15 12:00:00,2025-12-16 12:00:00,0.000000,-1.0,-0.000000
75%,NaN,2026-03-11 00:00:00,2026-03-12 00:00:00,0.015687,-1.0,0.020386
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.250000,-1.0,0.150000


In [142]:
stat_summary(strat_as[strat_as['long/short']==-1])

,ticker,start_date,date_after,return,long/short,profits
count,1752,1752,1752,1752.000000,1752.0,1752.000000
unique,248,NaN,NaN,NaN,NaN,NaN
top,CUAN,NaN,NaN,NaN,NaN,NaN
freq,37,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 18:47:40.273972,2025-12-24 07:01:38.630137,-0.000051,-1.0,0.000051
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.150000,-1.0,-0.250000
25%,NaN,2025-10-02 18:00:00,2025-10-05 06:00:00,-0.020386,-1.0,-0.015687
50%,NaN,2025-12-15 12:00:00,2025-12-16 12:00:00,0.000000,-1.0,-0.000000
75%,NaN,2026-03-11 00:00:00,2026-03-12 00:00:00,0.015687,-1.0,0.020386
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.250000,-1.0,0.150000


#### Strat Testing #2

Take top/bottom 20 percentile, long/short them. Percentile is based off three factors in which all of them is tested, namely: *avg_weighted_sentiment*, *total_sentiment*, *avg_sentiment*.

In [146]:
strat_aws_2 = strat_long_short(sentiment_df, indo_stock_returns, 'avg_weighted_sentiment',0.2)
strat_aws_2['profits'] = strat_aws_2['long/short']*strat_aws_2['return']
stat_summary(strat_aws_2)

,ticker,start_date,date_after,return,long/short,profits
count,7248,7248,7248,7248.000000,7248.000000,7248.000000
unique,319,NaN,NaN,NaN,NaN,NaN
top,ITMG,NaN,NaN,NaN,NaN,NaN
freq,100,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 17:42:30.993377,2025-12-24 06:00:47.682119,-0.000146,0.000000,0.000110
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.150000,-1.000000,-0.349315
25%,NaN,2025-10-03 00:00:00,2025-10-06 00:00:00,-0.019627,-1.000000,-0.017769
50%,NaN,2025-12-16 00:00:00,2025-12-17 00:00:00,0.000000,0.000000,0.000000
75%,NaN,2026-03-11 00:00:00,2026-03-12 00:00:00,0.014345,1.000000,0.017094
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.349315,1.000000,0.343511


In [147]:
strat_as_2 = strat_long_short(sentiment_df, indo_stock_returns, 'avg_sentiment',0.2)
strat_as_2['profits'] = strat_as_2['long/short']*strat_as_2['return']
stat_summary(strat_as_2)

,ticker,start_date,date_after,return,long/short,profits
count,7248,7248,7248,7248.000000,7248.000000,7248.000000
unique,316,NaN,NaN,NaN,NaN,NaN
top,ITMG,NaN,NaN,NaN,NaN,NaN
freq,97,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 17:42:30.993377,2025-12-24 06:00:47.682119,-0.000270,0.000000,0.000143
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.150000,-1.000000,-0.349315
25%,NaN,2025-10-03 00:00:00,2025-10-06 00:00:00,-0.019868,-1.000000,-0.018088
50%,NaN,2025-12-16 00:00:00,2025-12-17 00:00:00,-0.001145,0.000000,-0.000000
75%,NaN,2026-03-11 00:00:00,2026-03-12 00:00:00,0.014706,1.000000,0.017197
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.349315,1.000000,0.343511


In [148]:
strat_ts = strat_long_short(sentiment_df, indo_stock_returns, 'total_sentiment',0.2)
strat_ts['profits'] = strat_ts['long/short']*strat_ts['return']
stat_summary(strat_ts)

,ticker,start_date,date_after,return,long/short,profits
count,7248,7248,7248,7248.000000,7248.000000,7248.000000
unique,292,NaN,NaN,NaN,NaN,NaN
top,BBRI,NaN,NaN,NaN,NaN,NaN
freq,198,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 17:42:30.993377,2025-12-24 06:00:47.682119,-0.000799,0.000000,-0.000244
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.150000,-1.000000,-0.349315
25%,NaN,2025-10-03 00:00:00,2025-10-06 00:00:00,-0.018519,-1.000000,-0.016722
50%,NaN,2025-12-16 00:00:00,2025-12-17 00:00:00,-0.002148,0.000000,0.000000
75%,NaN,2026-03-11 00:00:00,2026-03-12 00:00:00,0.013307,1.000000,0.016129
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.349315,1.000000,0.248731


#### EWMA Strat

In [ ]:
sentiment = sentiment_df.copy()

sentiment["date"] = pd.to_datetime(
    sentiment["date"]
).dt.normalize()

sentiment["ticker"] = (
    sentiment["ticker"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Combine any duplicate ticker-date rows first
sentiment = (
    sentiment.groupby(
        ["date", "ticker"],
        as_index=False,
    )
    .agg(
        mention_count=("mention_count", "sum"),
        avg_sentiment=("avg_sentiment", "mean"),
        avg_weighted_sentiment=(
            "avg_weighted_sentiment",
            "mean",
        ),
    )
)


trading_dates = pd.DatetimeIndex(
    indo_stock_returns.index
).normalize()

trading_dates = trading_dates[
    (trading_dates >= sentiment["date"].min())
    & (trading_dates <= sentiment["date"].max())
]

tickers = sentiment["ticker"].unique()


full_index = pd.MultiIndex.from_product(
    [trading_dates, tickers],
    names=["date", "ticker"],
)

sentiment_filled = (
    sentiment
    .set_index(["date", "ticker"])
    .reindex(full_index)
    .fillna(0)
    .reset_index()
)

In [178]:
sentiment_aws = make_ewma(sentiment_filled)

strat_aws_3 = strat_long_short(sentiment_aws, indo_stock_returns, 'ewma_sentiment',0.1)
strat_aws_3['profits'] = strat_aws_3['long/short']*strat_aws_3['return']
stat_summary(strat_aws_3)

,ticker,start_date,date_after,return,long/short,profits
count,14520,14520,14520,14514.000000,14520.000000,14514.000000
unique,305,NaN,NaN,NaN,NaN,NaN
top,BRIS,NaN,NaN,NaN,NaN,NaN
freq,206,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 13:57:49.090909,2025-12-24 02:50:10.909091,-0.000128,0.000000,-0.000561
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.166667,-1.000000,-0.350000
25%,NaN,2025-09-30 18:00:00,2025-10-01 18:00:00,-0.018112,-1.000000,-0.016437
50%,NaN,2025-12-16 12:00:00,2025-12-17 12:00:00,0.000000,0.000000,-0.000000
75%,NaN,2026-03-12 06:00:00,2026-03-13 18:00:00,0.013986,1.000000,0.016129
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.350000,1.000000,0.250000


In [179]:
sentiment_as = make_ewma(sentiment_filled, 'avg_sentiment')

strat_as_3 = strat_long_short(sentiment_as, indo_stock_returns, 'ewma_sentiment',0.1)
strat_as_3['profits'] = strat_as_3['long/short']*strat_as_3['return']
stat_summary(strat_as_3)

,ticker,start_date,date_after,return,long/short,profits
count,14520,14520,14520,14513.000000,14520.000000,14513.000000
unique,302,NaN,NaN,NaN,NaN,NaN
top,BRIS,NaN,NaN,NaN,NaN,NaN
freq,204,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 13:57:49.090909,2025-12-24 02:50:10.909091,-0.000118,0.000000,-0.000452
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.166667,-1.000000,-0.350000
25%,NaN,2025-09-30 18:00:00,2025-10-01 18:00:00,-0.018003,-1.000000,-0.016158
50%,NaN,2025-12-16 12:00:00,2025-12-17 12:00:00,0.000000,0.000000,0.000000
75%,NaN,2026-03-12 06:00:00,2026-03-13 18:00:00,0.014085,1.000000,0.016181
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.350000,1.000000,0.250000


In [183]:
sentiment_filled['total_sentiment'] = sentiment_filled['avg_sentiment']*sentiment_filled['mention_count']

In [184]:
sentiment_ts = make_ewma(sentiment_filled, 'total_sentiment')

sentiment_ts_3 = strat_long_short(sentiment_ts, indo_stock_returns, 'ewma_sentiment',0.1)
sentiment_ts_3['profits'] = sentiment_ts_3['long/short']*sentiment_ts_3['return']
stat_summary(sentiment_ts_3)

,ticker,start_date,date_after,return,long/short,profits
count,14520,14520,14520,14511.000000,14520.000000,14511.000000
unique,295,NaN,NaN,NaN,NaN,NaN
top,BRIS,NaN,NaN,NaN,NaN,NaN
freq,220,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 13:57:49.090909,2025-12-24 02:50:10.909091,0.000180,0.000000,-0.000012
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.150000,-1.000000,-0.350000
25%,NaN,2025-09-30 18:00:00,2025-10-01 18:00:00,-0.017563,-1.000000,-0.015595
50%,NaN,2025-12-16 12:00:00,2025-12-17 12:00:00,0.000000,0.000000,0.000000
75%,NaN,2026-03-12 06:00:00,2026-03-13 18:00:00,0.013953,1.000000,0.016304
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.350000,1.000000,0.250000


In [187]:
sentiment_aws_2 = make_ewma(sentiment_filled)

strat_aws_4 = reverse_strat_long_short(sentiment_aws_2, indo_stock_returns, 'ewma_sentiment',0.1)
strat_aws_4['profits'] = strat_aws_4['long/short']*strat_aws_4['return']
stat_summary(strat_aws_4)

,ticker,start_date,date_after,return,long/short,profits
count,14520,14520,14520,14514.000000,14520.000000,14514.000000
unique,305,NaN,NaN,NaN,NaN,NaN
top,BRIS,NaN,NaN,NaN,NaN,NaN
freq,206,NaN,NaN,NaN,NaN,NaN
mean,NaN,2025-12-22 13:57:49.090909,2025-12-24 02:50:10.909091,-0.000128,0.000000,0.000561
min,NaN,2025-07-14 00:00:00,2025-07-15 00:00:00,-0.166667,-1.000000,-0.250000
25%,NaN,2025-09-30 18:00:00,2025-10-01 18:00:00,-0.018112,-1.000000,-0.016129
50%,NaN,2025-12-16 12:00:00,2025-12-17 12:00:00,0.000000,0.000000,0.000000
75%,NaN,2026-03-12 06:00:00,2026-03-13 18:00:00,0.013986,1.000000,0.016437
max,NaN,2026-06-15 00:00:00,2026-06-17 00:00:00,0.350000,1.000000,0.350000
